In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import mplhep as hep
from matplotlib.patches import Rectangle
from pathlib import Path
from typing import List, Union
from scipy.linalg import solve_triangular, qr
import json
import re
import os
import uproot
import importlib.util
plt.style.use(hep.style.CMS)

In [ ]:
label_map_nj = {
     'nj_bin0': 'r_NJ_0p0_1p0',
 'nj_bin1': 'r_NJ_1p0_2p0',
 'nj_bin2': 'r_NJ_2p0_3p0',
 'nj_bin3': 'r_NJ_3p0_4p0',
 'nj_bin4': 'r_NJ_4p0_100p0',
}


pois = {
    "PTH": ["r_PTH_0p0_5p0","r_PTH_5p0_10p0","r_PTH_10p0_15p0","r_PTH_15p0_20p0","r_PTH_20p0_25p0","r_PTH_25p0_30p0","r_PTH_30p0_35p0","r_PTH_35p0_45p0","r_PTH_45p0_60p0","r_PTH_60p0_80p0","r_PTH_80p0_100p0","r_PTH_100p0_120p0","r_PTH_120p0_140p0","r_PTH_140p0_170p0","r_PTH_170p0_200p0","r_PTH_200p0_250p0","r_PTH_250p0_350p0","r_PTH_350p0_450p0","r_PTH_450p0_10000p0"],
    "NJ": ["r_NJ_0p0_1p0", "r_NJ_1p0_2p0", "r_NJ_2p0_3p0", "r_NJ_3p0_4p0", "r_NJ_4p0_100p0"],
    "PTJ0": ["r_PTJ0_m10000p0_30p0", "r_PTJ0_30p0_40p0", "r_PTJ0_40p0_55p0", "r_PTJ0_55p0_75p0", "r_PTJ0_75p0_95p0", "r_PTJ0_95p0_120p0", "r_PTJ0_120p0_150p0", "r_PTJ0_150p0_200p0", "r_PTJ0_200p0_10000p0"],
    "YH": ["r_YH_0p0_0p15", "r_YH_0p15_0p3", "r_YH_0p3_0p45", "r_YH_0p45_0p6", "r_YH_0p6_0p75", "r_YH_0p75_0p9", "r_YH_0p9_1p2", "r_YH_1p2_1p6", "r_YH_1p6_2p0", "r_YH_2p0_2p5"]
}

In [ ]:
palette = sns.diverging_palette(240, 10, n=20, as_cmap=True)


# Plot correlation matrix at gen-level
def plot_corr(_corr_gen, title="Correlation Matrix (Generator Level)", save_path=None, label_map=None):
    corr_gen_mask = np.triu(np.ones_like(_corr_gen, dtype=bool), k=1)
    fig_corr_gen, ax_corr_gen = plt.subplots(figsize=(28, 18), constrained_layout=True)
    sns.heatmap(
        _corr_gen,
        ax=ax_corr_gen,
        cmap=palette,
        mask=corr_gen_mask,
        vmin=-1,
        vmax=1,
        annot=True,
        fmt=".2f",
        center=0,
        square=True,
        annot_kws={"size": 10, "weight": "bold"},
        cbar_kws={"label": "Correlation"},
    )
    ax_corr_gen.set_title(title)
    ax_corr_gen.set_xlabel("Bins")
    ax_corr_gen.set_ylabel("Bins")
    
    # ---- LABEL TRANSLATION LOGIC ----
    if label_map is not None:
        # assume index == columns order
        original_labels = list(_corr_gen.index)

        translated_labels = [
            label_map.get(lbl, lbl) for lbl in original_labels
        ]

        ax_corr_gen.set_xticklabels(translated_labels, rotation=90, fontsize=12)
        ax_corr_gen.set_yticklabels(translated_labels, rotation=0, fontsize=12)
        # IMPORTANT: re-apply tick styling AFTER setting labels
        ax_corr_gen.tick_params(
            axis="x",
            which="both",
            length=0,   # <- removes tick marks
            bottom=False,
            top=False
        )
        ax_corr_gen.tick_params(
            axis="y",
            which="both",
            length=0,   # <- removes tick marks
            left=False,
            right=False
        )

    else:
        ax_corr_gen.tick_params(
            axis="x", which="both", labelrotation=90, bottom=False, top=False, length=0, labelsize=12
        )
        ax_corr_gen.tick_params(
            axis="y", which="both", labelrotation=0, left=False, right=False, length=0, labelsize=12
        )

    ax_corr_gen.minorticks_off()

    if save_path:
        plt.savefig(save_path, dpi=300)
    else:
        plt.show()

# Plot covariance matrix at gen-level
def plot_cov(_corr_gen, title="Covariance Matrix (Generator Level)", save_path=None, label_map=None):
    corr_gen_mask = np.triu(np.ones_like(_corr_gen, dtype=bool), k=1)
    fig_corr_gen, ax_corr_gen = plt.subplots(figsize=(28, 18), constrained_layout=True)
    sns.heatmap(
        _corr_gen,
        ax=ax_corr_gen,
        cmap=palette,
        mask=corr_gen_mask,
        vmin = _corr_gen.to_numpy().min(),
        vmax = _corr_gen.to_numpy().max(),
        annot=True,
        fmt=".2f",
        center=0,
        square=True,
        annot_kws={"size": 10, "weight": "bold"},
        cbar_kws={"label": r"Theoretical Covariance [fb$^{2}$]"},
        # cbar_kws={"label": "Theoretical Covariance"},
    )
    ax_corr_gen.set_title(title)
    ax_corr_gen.set_xlabel("Bins")
    ax_corr_gen.set_ylabel("Bins")
    
    # ---- LABEL TRANSLATION LOGIC ----
    if label_map is not None:
        # assume index == columns order
        original_labels = list(_corr_gen.index)

        translated_labels = [
            label_map.get(lbl, lbl) for lbl in original_labels
        ]

        ax_corr_gen.set_xticklabels(translated_labels, rotation=90, fontsize=12)
        ax_corr_gen.set_yticklabels(translated_labels, rotation=0, fontsize=12)
        # IMPORTANT: re-apply tick styling AFTER setting labels
        ax_corr_gen.tick_params(
            axis="x",
            which="both",
            length=0,   # <- removes tick marks
            bottom=False,
            top=False
        )
        ax_corr_gen.tick_params(
            axis="y",
            which="both",
            length=0,   # <- removes tick marks
            left=False,
            right=False
        )

    else:
        ax_corr_gen.tick_params(
            axis="x", which="both", labelrotation=90, bottom=False, top=False, length=0, labelsize=12
        )
        ax_corr_gen.tick_params(
            axis="y", which="both", labelrotation=0, left=False, right=False, length=0, labelsize=12
        )

    ax_corr_gen.minorticks_off()

    if save_path:
        plt.savefig(save_path, dpi=300)
    else:
        plt.show()


def corr_from_cov(cov_mat: pd.DataFrame) -> pd.DataFrame:
    v = np.diag(cov_mat.to_numpy())
    sigma = np.sqrt(np.clip(v, 0.0, None))
    with np.errstate(divide="ignore", invalid="ignore"):
        outer = np.outer(1.0 / sigma, 1.0 / sigma)
        arr = outer * cov_mat.to_numpy()
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    np.fill_diagonal(arr, 1.0)
    return pd.DataFrame(arr, index=cov_mat.index, columns=cov_mat.columns)


In [4]:
# For the moment only using NJ. I wanna see first if it is working....

FILE_RE = re.compile(
    r"^fidXS_(?P<obs>NJ)_all_exclusive\.py$"
)

def load_python_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load file: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

var = {
    "scale": {
        "NJ": [],
    }
}

fidXS_folder = Path("fidXS")

for path in fidXS_folder.iterdir():
    match = FILE_RE.match(path.name)
    if not match:
        continue

    obs = match.group("obs")

    module = load_python_module(path)

    if not hasattr(module, "fidXS"):
        print(f"Skipping {path.name}: no fidXS vector")
        continue

    fid_xs_nom = np.array(list(module.fidXS))  # convert once to numpy
    fid_xs_scale_up = np.array(list(module.fidXS_scale_up))
    
    for i, nom_xs in enumerate(fid_xs_nom):
        var["scale"][obs].append(fid_xs_scale_up[i] - fid_xs_nom[i])



# We fully correlate the theory uncertainties (WRONG APPROACH)

# Build the correlation matrix
n_bins = len(var["scale"]["NJ"])
matrix = np.ones((5, 5), dtype=int)

# Use the dictionary values (or keys) as labels
labels = list(label_map_nj.values())

correlation = pd.DataFrame(
    matrix,
    index=labels,
    columns=labels
)

correlation = correlation * 1

# Build the global ordered list of bins
all_bins = []
all_bins.extend(pois["NJ"])

# Replace the nested dicts with column-vector DataFrames
for source in var:

    values = []

    values.extend(var[source]["NJ"])

    var[source] = pd.DataFrame(
        values,
        index=all_bins,
        columns=[source]
    )


# Now compute the covariance per source

v_scale = var["scale"].iloc[:, 0]
Cov_scale_NJ = correlation.mul(v_scale, axis=0).mul(v_scale, axis=1)

# plot_corr(Cov_scale_NJ, title="Covariance (No Tackmann)")
Cov_scale_NJ

,r_NJ_0p0_1p0,r_NJ_1p0_2p0,r_NJ_2p0_3p0,r_NJ_3p0_4p0,r_NJ_4p0_100p0
r_NJ_0p0_1p0,NaN,NaN,NaN,NaN,NaN
r_NJ_1p0_2p0,NaN,NaN,NaN,NaN,NaN
r_NJ_2p0_3p0,NaN,NaN,NaN,NaN,NaN
r_NJ_3p0_4p0,NaN,NaN,NaN,NaN,NaN
r_NJ_4p0_100p0,NaN,NaN,NaN,NaN,NaN


In [5]:
def long_range_st(sigma, sigma_up, sigma_dn):
    sigma = np.array(sigma)
    up = np.array(sigma_up)
    dn = np.array(sigma_dn)

    nbins = len(sigma)

    # --------------------------------------------------
    # 1. absolute uncertainties (envelope)
    # --------------------------------------------------
    Delta = np.maximum(np.abs(up - sigma), np.abs(dn - sigma))

    # relative uncertainties
    delta = Delta / np.clip(sigma, 1e-12, None)

    # --------------------------------------------------
    # 2. covariance construction
    # --------------------------------------------------
    Cov = np.zeros((nbins, nbins))

    # --- normalization-like component (NP total)
    v_norm = Delta
    Cov += np.outer(v_norm, v_norm)

    # --------------------------------------------------
    # 3. migration components (long-range ST)
    # --------------------------------------------------
    for i in range(nbins - 1):
        hi = i + 1

        # migration size (from slide)
        sigma_hi = sigma[hi]
        sigma_lo = sigma[i]

        if sigma_hi <= 0 or sigma_lo <= 0:
            continue

        # relative migration
        d_hi = Delta[hi] / sigma_hi

        # construct migration vector
        v = np.zeros(nbins)

        # + in high bin
        v[hi] = +d_hi * sigma_hi

        # - in low bin (compensation)
        v[i] = -d_hi * sigma_hi * (sigma_hi / sigma_lo)
        
        print(v)

        Cov += np.outer(v, v)

    return Cov

In [6]:
FILE_RE = re.compile(
    r"^fidXS_(?P<obs>NJ)_all_inclusive\.py$"
)

def load_python_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load file: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

var = {
    "scale": {
        "NJ": [],
    }
}

fidXS_folder = Path("fidXS")

for path in fidXS_folder.iterdir():
    match = FILE_RE.match(path.name)
    if not match:
        continue

    obs = match.group("obs")

    module = load_python_module(path)

    if not hasattr(module, "fidXS"):
        print(f"Skipping {path.name}: no fidXS vector")
        continue

    sigma = np.array(list(module.fidXS))  # convert once to numpy
    up = np.array(list(module.fidXS_scale_up))
    dn = np.array(list(module.fidXS_scale_dn))
    

    Delta_up = up - sigma
    Delta_dn = dn - sigma

    # # envelope (standard theory choice)
    # Delta_ge = np.maximum(np.abs(Delta_up), np.abs(Delta_dn))
    
    # sigma_excl = np.zeros_like(sigma)

    # sigma_excl[0] = sigma[0] - sigma[1]
    # sigma_excl[1] = sigma[1] - sigma[2]
    # sigma_excl[2] = sigma[2] - sigma[3]
    # sigma_excl[3] = sigma[3] - sigma[4]
    # sigma_excl[4] = sigma[4]
    
    # modes = []

    # # ≥0 mode
    # v0 = np.array([1,1,1,1,1]) * Delta_ge[0]
    # modes.append(v0)

    # # ≥1 mode
    # v1 = np.array([0,1,1,1,1]) * Delta_ge[1]
    # modes.append(v1)

    # # ≥2 mode
    # v2 = np.array([0,0,1,1,1]) * Delta_ge[2]
    # modes.append(v2)

    # # ≥3 mode
    # v3 = np.array([0,0,0,1,1]) * Delta_ge[3]
    # modes.append(v3)

    # # ≥4 mode
    # v4 = np.array([0,0,0,0,1]) * Delta_ge[4]
    # modes.append(v4)
    
    # Cov = np.zeros((5,5))
    
    # for v in modes:
    #     Cov += np.outer(v, v)
    Cov = long_range_st(sigma, up, dn)
        
    # print(corr_from_cov(pd.DataFrame(Cov)))
    print(Cov)
    
# # 4. covariance
# V = np.vstack([Delta_0, Delta_1, Delta_2, Delta_3, Delta_4])
# Cov_scale_NJ = V @ V.T

# corr_from_cov(pd.DataFrame(Cov_scale_NJ))
# # plot_corr(Cov_scale_NJ, title="Covariance (Tackmann)")

In [7]:
### Kljinsma method

In [8]:
FILE_RE = re.compile(
    r"^fidXS_(?P<obs>PTH|NJ|PTJ0|YH)_all_scale(?P<scale>0|1|3|5|7|8)\.py$"
)

def load_python_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load file: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

scale_variations = {
    "0": {
        "PTH": [],
        "NJ": [],
        "PTJ0": [],
        "YH": [],
    },
    "1": {
        "PTH": [],
        "NJ": [],
        "PTJ0": [],
        "YH": [],
    },
    "3": {
        "PTH": [],
        "NJ": [],
        "PTJ0": [],
        "YH": [],
    },
    "5": {
        "PTH": [],
        "NJ": [],
        "PTJ0": [],
        "YH": [],
    },
    "7": {
        "PTH": [],
        "NJ": [],
        "PTJ0": [],
        "YH": [],
    },
    "8": {
        "PTH": [],
        "NJ": [],
        "PTJ0": [],
        "YH": [],
    }
}

fidXS_folder = Path("fidXS")

for path in fidXS_folder.iterdir():
    match = FILE_RE.match(path.name)
    if not match:
        continue

    obs = match.group("obs")
    scale = match.group("scale")

    module = load_python_module(path)

    if not hasattr(module, "fidXS"):
        print(f"Skipping {path.name}: no fidXS vector")
        continue

    sigma = np.array(list(module.fidXS))  # convert once to numpy
    up = np.array(list(module.fidXS_scale_up))
    
    scale_variations[scale][obs] = up - sigma
    
    


In [9]:
scale_variations

{'0': {'PTH': array([-0.14293521, -0.24610112, -0.22992654, -0.14792944, -0.04984755,
          0.0404134 ,  0.14022355,  0.39287711,  0.57765465,  0.5339604 ,
          0.36812889,  0.23453765,  0.14852689,  0.1685265 ,  0.10741393,
          0.10548222,  0.09747405,  0.0282445 ,  0.01970366]),
  'NJ': array([-0.66247604,  1.42401925,  0.98321418,  0.27339047,  0.12842843]),
  'PTJ0': array([-0.66247604,  0.54436361,  0.6394488 ,  0.53385421,  0.32899367,
          0.2631986 ,  0.17828142,  0.14636522, -0.48792923]),
  'YH': array([0.20144103, 0.19806995, 0.19286369, 0.1964478 , 0.17263808,
         0.17831714, 0.30280625, 0.3388179 , 0.25226262, 0.11291182])},
 '1': {'PTH': array([-0.01832525, -0.02716566, -0.01974745,  0.00499766,  0.04024493,
          0.08105802,  0.12831718,  0.32761086,  0.48025162,  0.44009588,
          0.28234024,  0.17099154,  0.10409639,  0.10806378,  0.06303978,
          0.05963593,  0.05180578,  0.01548302,  0.00992864]),
  'NJ': array([0.15138361, 1.152

In [10]:
import numpy as np

for obs in ["NJ"]:

    n_bins = 5
    corr = np.zeros((n_bins, n_bins))
    cov = np.zeros((n_bins, n_bins))

    for bin_a in range(n_bins):
        for bin_b in range(n_bins):

            nom = 0.0
            denom_a = 0.0
            denom_b = 0.0

            for i in scale_variations:

                xa = scale_variations[i][obs][bin_a]
                xb = scale_variations[i][obs][bin_b]

                nom += xa * xb
                denom_a += xa**2
                denom_b += xb**2

            corr[bin_a, bin_b] = nom / np.sqrt(denom_a * denom_b)

    print(f"Correlation matrix for {obs}:")
    print(corr)
    

Correlation matrix for NJ:
[[ 1.          0.0202423  -0.40062668 -0.4007645  -0.16985719]
 [ 0.0202423   1.          0.90737147  0.90602509  0.97449491]
 [-0.40062668  0.90737147  1.          0.99635161  0.96025821]
 [-0.4007645   0.90602509  0.99635161  1.          0.96919583]
 [-0.16985719  0.97449491  0.96025821  0.96919583  1.        ]]


In [11]:
import numpy as np

for obs in ["NJ"]:

    n_bins = 5

    cov = np.zeros((n_bins, n_bins))
    corr = np.zeros((n_bins, n_bins))

    for bin_a in range(n_bins):
        for bin_b in range(n_bins):

            xa = np.array([
                scale_variations[i][obs][bin_a]
                for i in scale_variations
            ])

            xb = np.array([
                scale_variations[i][obs][bin_b]
                for i in scale_variations
            ])

            # subtract means
            xa_cent = xa - xa.mean()
            xb_cent = xb - xb.mean()

            cov_ab = np.mean(xa_cent * xb_cent)

            cov[bin_a, bin_b] = cov_ab

    # correlation matrix
    diag = np.sqrt(np.diag(cov))

    for bin_a in range(n_bins):
        for bin_b in range(n_bins):
            corr[bin_a, bin_b] = (
                cov[bin_a, bin_b]
                / (diag[bin_a] * diag[bin_b])
            )

    print(f"\nObservable: {obs}")

    print("\nCovariance matrix:")
    print(cov)

    print("\nCorrelation matrix:")
    print(corr)


Observable: NJ

Covariance matrix:
[[ 2.21620985 -0.85272177 -0.80657201 -0.22274136 -0.0894723 ]
 [-0.85272177  0.32938524  0.31171972  0.085592    0.03421485]
 [-0.80657201  0.31171972  0.29513788  0.08095618  0.03232495]
 [-0.22274136  0.085592    0.08095618  0.02239974  0.00901085]
 [-0.0894723   0.03421485  0.03232495  0.00901085  0.00364876]]

Correlation matrix:
[[ 1.         -0.99804435 -0.99729865 -0.99970967 -0.99497089]
 [-0.99804435  1.          0.99976915  0.99645936  0.98693774]
 [-0.99729865  0.99976915  1.          0.99567137  0.98503719]
 [-0.99970967  0.99645936  0.99567137  1.          0.99671566]
 [-0.99497089  0.98693774  0.98503719  0.99671566  1.        ]]


In [12]:
label_map = {
    "PTH_bin0": "r_PTH_0p0_5p0",
    "PTH_bin1": "r_PTH_5p0_10p0",
    "PTH_bin2": "r_PTH_10p0_15p0",
    "PTH_bin3": "r_PTH_15p0_20p0",
    "PTH_bin4": "r_PTH_20p0_25p0",
    "PTH_bin5": "r_PTH_25p0_30p0",
    "PTH_bin6": "r_PTH_30p0_35p0",
    "PTH_bin7": "r_PTH_35p0_45p0",
    "PTH_bin8": "r_PTH_45p0_60p0",
    "PTH_bin9": "r_PTH_60p0_80p0",
    "PTH_bin10": "r_PTH_80p0_100p0",
    "PTH_bin11": "r_PTH_100p0_120p0",
    "PTH_bin12": "r_PTH_120p0_140p0",
    "PTH_bin13": "r_PTH_140p0_170p0",
    "PTH_bin14": "r_PTH_170p0_200p0",
    "PTH_bin15": "r_PTH_200p0_250p0",
    "PTH_bin16": "r_PTH_250p0_350p0",
    "PTH_bin17": "r_PTH_350p0_450p0",
    "PTH_bin18": "r_PTH_450p0_10000p0",
    "NJ_bin0": "r_NJ_0p0_1p0",
    "NJ_bin1": "r_NJ_1p0_2p0",
    "NJ_bin2": "r_NJ_2p0_3p0",
    "NJ_bin3": "r_NJ_3p0_4p0",
    "NJ_bin4": "r_NJ_4p0_100p0",
    "PTJ0_bin0": "r_PTJ0_m10000p0_30p0",
    "PTJ0_bin1": "r_PTJ0_30p0_40p0",
    "PTJ0_bin2": "r_PTJ0_40p0_55p0",
    "PTJ0_bin3": "r_PTJ0_55p0_75p0",
    "PTJ0_bin4": "r_PTJ0_75p0_95p0",
    "PTJ0_bin5": "r_PTJ0_95p0_120p0",
    "PTJ0_bin6": "r_PTJ0_120p0_150p0",
    "PTJ0_bin7": "r_PTJ0_150p0_200p0",
    "PTJ0_bin8": "r_PTJ0_200p0_10000p0",
    "YH_bin0": "r_YH_0p0_0p15",
    "YH_bin1": "r_YH_0p15_0p3",
    "YH_bin2": "r_YH_0p3_0p45",
    "YH_bin3": "r_YH_0p45_0p6",
    "YH_bin4": "r_YH_0p6_0p75",
    "YH_bin5": "r_YH_0p75_0p9",
    "YH_bin6": "r_YH_0p9_1p2",
    "YH_bin7": "r_YH_1p2_1p6",
    "YH_bin8": "r_YH_1p6_2p0",
    "YH_bin9": "r_YH_2p0_2p5",
}

In [13]:
import numpy as np
import pandas as pd

# observables = ["PTH", "NJ", "PTJ0", "YH"]
observables = ["NJ"]
bin_info = {
    # "PTH": 19,
    "NJ": 5,
    # "PTJ0": 9,
    # "YH": 10
}

# -----------------------------
# 1. Build column labels
# -----------------------------
cols = []
for obs in observables:
    for b in range(bin_info[obs]):
        cols.append(f"{obs}_bin{b}")

# -----------------------------
# 2. Build data frame
# -----------------------------
data = []

for i in scale_variations:
    row = []
    for obs in observables:
        for b in range(bin_info[obs]):
            row.append(scale_variations[i][obs][b])
    data.append(row)

df = pd.DataFrame(data, columns=cols)

# -----------------------------
# 3. Mean subtraction (important!)
# -----------------------------
df_centered = df - df.mean()

df_centered= df_centered.rename(index=label_map, columns=label_map)

# -----------------------------
# 4. Covariance matrix
# -----------------------------
cov = df_centered.cov(ddof=0)   # same as bias=True in numpy

# -----------------------------
# 5. Correlation matrix
# -----------------------------
corr = df_centered.corr()

print("\nCovariance matrix:")
print(cov)

print("\nCorrelation matrix:")
print(corr)


Covariance matrix:
                r_NJ_0p0_1p0  r_NJ_1p0_2p0  r_NJ_2p0_3p0  r_NJ_3p0_4p0  \
r_NJ_0p0_1p0        2.216210     -0.852722     -0.806572     -0.222741   
r_NJ_1p0_2p0       -0.852722      0.329385      0.311720      0.085592   
r_NJ_2p0_3p0       -0.806572      0.311720      0.295138      0.080956   
r_NJ_3p0_4p0       -0.222741      0.085592      0.080956      0.022400   
r_NJ_4p0_100p0     -0.089472      0.034215      0.032325      0.009011   

                r_NJ_4p0_100p0  
r_NJ_0p0_1p0         -0.089472  
r_NJ_1p0_2p0          0.034215  
r_NJ_2p0_3p0          0.032325  
r_NJ_3p0_4p0          0.009011  
r_NJ_4p0_100p0        0.003649  

Correlation matrix:
                r_NJ_0p0_1p0  r_NJ_1p0_2p0  r_NJ_2p0_3p0  r_NJ_3p0_4p0  \
r_NJ_0p0_1p0        1.000000     -0.998044     -0.997299     -0.999710   
r_NJ_1p0_2p0       -0.998044      1.000000      0.999769      0.996459   
r_NJ_2p0_3p0       -0.997299      0.999769      1.000000      0.995671   
r_NJ_3p0_4p0       -

In [14]:
import numpy as np
import pandas as pd

observables = ["PTH", "NJ", "PTJ0", "YH"]
bin_info = {
    "PTH": 19,
    "NJ": 5,
    "PTJ0": 9,
    "YH": 10
}

# -----------------------------
# Build DataFrame (your setup)
# -----------------------------
cols = []
for obs in observables:
    for b in range(bin_info[obs]):
        cols.append(f"{obs}_bin{b}")

data = []

for i in scale_variations:
    row = []
    for obs in observables:
        for b in range(bin_info[obs]):
            row.append(scale_variations[i][obs][b])
    data.append(row)

df = pd.DataFrame(data, columns=cols)

# -----------------------------
# Cosine similarity matrix (UNCHANGED, uncentered)
# -----------------------------
bins = df.columns
n_bins = len(bins)

corr = pd.DataFrame(
    np.zeros((n_bins, n_bins)),
    index=bins,
    columns=bins
)

for a in bins:
    xa = df[a].to_numpy()

    denom_a = np.sum(xa ** 2)

    for b in bins:
        xb = df[b].to_numpy()

        denom_b = np.sum(xb ** 2)
        num = np.sum(xa * xb)

        corr.loc[a, b] = num / np.sqrt(denom_a * denom_b)

print("Uncentered correlation (cosine similarity):")
print(corr)

Uncentered correlation (cosine similarity):
           PTH_bin0  PTH_bin1  PTH_bin2  PTH_bin3  PTH_bin4  PTH_bin5  \
PTH_bin0   1.000000  0.999392  0.998582  0.992720  0.967846  0.882060   
PTH_bin1   0.999392  1.000000  0.999824  0.996247  0.975713  0.896763   
PTH_bin2   0.998582  0.999824  1.000000  0.997686  0.979604  0.904691   
PTH_bin3   0.992720  0.996247  0.997686  1.000000  0.990976  0.931226   
PTH_bin4   0.967846  0.975713  0.979604  0.990976  1.000000  0.971443   
PTH_bin5   0.882060  0.896763  0.904691  0.931226  0.971443  1.000000   
PTH_bin6   0.572384  0.597781  0.612107  0.663364  0.756813  0.890232   
PTH_bin7   0.133471  0.162868  0.180089  0.243744  0.368630  0.577946   
PTH_bin8  -0.196707 -0.169190 -0.152540 -0.089647  0.039167  0.272652   
PTH_bin9  -0.323185 -0.296961 -0.280907 -0.219913 -0.093008  0.143260   
PTH_bin10 -0.403260 -0.376922 -0.360950 -0.300322 -0.173665  0.063921   
PTH_bin11 -0.426242 -0.399275 -0.383086 -0.321965 -0.194609  0.043468   
PTH_bin

In [15]:
# Now compute the scale uncertainties from the fidXS output

FILE_RE = re.compile(
    r"^fidXS_(?P<obs>PTH|NJ|PTJ0|YH)_all\.py$"
)

def load_python_module(path: Path):
    spec = importlib.util.spec_from_file_location(path.stem, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"Cannot load file: {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

scale_uncertainties = {
    "PTH": [],
    "NJ": [],
    "PTJ0": [],
    "YH": [],
}

fidXS_folder = Path("fidXS")

for path in fidXS_folder.iterdir():
    match = FILE_RE.match(path.name)
    if not match:
        continue

    obs = match.group("obs")
    module = load_python_module(path)

    if not hasattr(module, "fidXS"):
        print(f"Skipping {path.name}: no fidXS vector")
        continue

    sigma = np.array(list(module.fidXS))  # convert once to numpy
    up = np.array(list(module.fidXS_scale_up))
    
    scale_uncertainties[obs] = up - sigma
    

In [16]:
unc = []

for obs in observables:
    unc.extend(scale_uncertainties[obs])  # your dict

sigma_THU = np.array(unc)

In [17]:
cov = corr * np.outer(sigma_THU, sigma_THU)

cov = pd.DataFrame(
    cov,
    index=df.columns,
    columns=df.columns
)

In [18]:
cov

,PTH_bin0,PTH_bin1,PTH_bin2,PTH_bin3,PTH_bin4,PTH_bin5,PTH_bin6,PTH_bin7,PTH_bin8,PTH_bin9,...,YH_bin0,YH_bin1,YH_bin2,YH_bin3,YH_bin4,YH_bin5,YH_bin6,YH_bin7,YH_bin8,YH_bin9
PTH_bin0,0.155237,0.293996,0.287932,0.234412,0.166476,0.099552,0.033721,0.021147,-0.045621,-0.069318,...,0.066128,0.066495,0.066867,0.059916,0.058281,0.049448,0.095833,0.100430,0.054414,0.024975
PTH_bin1,0.293996,0.557461,0.546311,0.445791,0.318036,0.191795,0.066737,0.048901,-0.074357,-0.120699,...,0.130000,0.130531,0.131220,0.117872,0.114332,0.097444,0.188331,0.197843,0.108206,0.049803
PTH_bin2,0.287932,0.546311,0.535571,0.437581,0.312973,0.189654,0.066981,0.052999,-0.065710,-0.111910,...,0.130031,0.130499,0.131162,0.117977,0.114284,0.097626,0.188431,0.198120,0.108900,0.050189
PTH_bin3,0.234412,0.445791,0.437581,0.359181,0.259279,0.159869,0.059446,0.058744,-0.031625,-0.071747,...,0.114089,0.114353,0.114855,0.103771,0.100099,0.086154,0.165551,0.174492,0.097499,0.045114
PTH_bin4,0.166476,0.318036,0.312973,0.259279,0.190588,0.121484,0.049403,0.064716,0.010065,-0.022104,...,0.093082,0.093172,0.093468,0.085072,0.081529,0.071017,0.135520,0.143269,0.082181,0.038247
PTH_bin5,0.099552,0.191795,0.189654,0.159869,0.121484,0.082055,0.038130,0.066575,0.045973,0.022340,...,0.070144,0.070219,0.070318,0.064638,0.061481,0.054373,0.102876,0.108906,0.064631,0.030268
PTH_bin6,0.033721,0.066737,0.066981,0.059446,0.049403,0.038130,0.022358,0.053215,0.059546,0.046686,...,0.039674,0.039732,0.039678,0.037030,0.034819,0.031505,0.058850,0.062414,0.038905,0.018371
PTH_bin7,0.021147,0.048901,0.052999,0.058744,0.064716,0.066575,0.053215,0.161713,0.222806,0.194554,...,0.091114,0.091664,0.091243,0.086500,0.080583,0.074487,0.137568,0.145380,0.095082,0.045159
PTH_bin8,-0.045621,-0.074357,-0.065710,-0.031625,0.010065,0.045973,0.059546,0.222806,0.346486,0.317597,...,0.098186,0.099566,0.098734,0.095199,0.087965,0.083056,0.151739,0.159147,0.109339,0.052154
PTH_bin9,-0.069318,-0.120699,-0.111910,-0.071747,-0.022104,0.022340,0.046686,0.194554,0.317597,0.296343,...,0.075123,0.076422,0.075607,0.073718,0.067679,0.064851,0.117580,0.123030,0.087180,0.041739


In [19]:
std = df.std(ddof=0)   # or ddof=1 depending on your convention

cov = corr * np.outer(std, std)

cov = pd.DataFrame(cov, index=df.columns, columns=df.columns)